In [1]:
# Setup
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pulp
import pandas as pd
import numpy as np
import mlflow

# ── Spark + MinIO ─────────────────────────────────────────────────
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("HospitalOptimizer-PuLP") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://hospital-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "hospital") \
    .config("spark.hadoop.fs.s3a.secret.key", "hospital123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark {spark.version} démarré")
print(f"✅ PuLP {pulp.__version__}")

✅ Spark 3.5.0 démarré
✅ PuLP 2.7.0


In [2]:
# Calcul de la charge par heure

# ── Charger les admissions depuis Silver ──────────────────────────
admissions = spark.read.parquet("s3a://silver/parquet/admissions.parquet").cache()

# ── Agréger les admissions par heure ─────────────────────────────
admissions_hourly = admissions \
    .withColumn("hour", F.hour("ADMITTIME")) \
    .groupBy("hour") \
    .agg(F.count("HADM_ID").alias("n_admissions")) \
    .orderBy("hour") \
    .toPandas()

print("✅ Charge par heure calculée")
print(admissions_hourly.to_string(index=False))

✅ Charge par heure calculée
 hour  n_admissions
    0          3389
    1          3327
    2          3405
    3          3357
    4          3355
    5          3349
    6          3360
    7          3280
    8          3253
    9          3302
   10          3347
   11          3356
   12          3333
   13          3385
   14          3350
   15          3423
   16          3300
   17          3323
   18          3348
   19          3221
   20          3326
   21          3301
   22          3290
   23          3320


In [4]:
# Modèle d'optimisation PuLP 

# ── Paramètres ────────────────────────────────────────────────────
TOTAL_NURSES = 20
MIN_NURSES   = 2
HOURS        = list(range(24))

# Normaliser la charge
max_admissions = admissions_hourly['n_admissions'].max()
charge = {
    row['hour']: row['n_admissions'] / max_admissions
    for _, row in admissions_hourly.iterrows()
}

# ── Problème d'optimisation ───────────────────────────────────────
prob = pulp.LpProblem("StaffingOptimization", pulp.LpMaximize)

# Variables : nombre d'infirmiers par heure
nurses = {
    h: pulp.LpVariable(f"nurses_{h}", lowBound=MIN_NURSES, cat='Integer')
    for h in HOURS
}

# ── Objectif : maximiser la couverture ───────────────────────────
# Plus d'infirmiers aux heures chargées = moins de temps d'attente
prob += pulp.lpSum(charge[h] * nurses[h] for h in HOURS)

# ── Contraintes ───────────────────────────────────────────────────
# Budget total d'infirmiers par jour
prob += pulp.lpSum(nurses[h] for h in HOURS) <= TOTAL_NURSES * 24

# Heures de pointe → minimum 3 infirmiers
peak_hours = admissions_hourly.nlargest(6, 'n_admissions')['hour'].tolist()
for h in peak_hours:
    prob += nurses[h] >= 3

# ── Résoudre ──────────────────────────────────────────────────────
prob.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"✅ Optimisation terminée")
print(f"   Statut : {pulp.LpStatus[prob.status]}")

✅ Optimisation terminée
   Statut : Optimal


In [5]:
# ── Extraire la solution ──────────────────────────────────────────
results = []
for h in HOURS:
    n = int(pulp.value(nurses[h]))
    results.append({
        'heure':        h,
        'infirmiers':   n,
        'admissions':   admissions_hourly[admissions_hourly['hour']==h]['n_admissions'].values[0],
        'charge':       round(charge[h], 3),
        'est_pic':      h in peak_hours
    })

df_results = pd.DataFrame(results)

# ── Calcul du KPI ─────────────────────────────────────────────────
# Temps d'attente = admissions / infirmiers
df_results['temps_attente_avant'] = df_results['admissions'] / MIN_NURSES
df_results['temps_attente_apres'] = df_results['admissions'] / df_results['infirmiers']

temps_avant = df_results['temps_attente_avant'].mean()
temps_apres = df_results['temps_attente_apres'].mean()
reduction   = (temps_avant - temps_apres) / temps_avant * 100

print("=" * 55)
print("RÉSULTATS OPTIMISATION STAFFING")
print("=" * 55)
print(f"\nTemps attente avant : {temps_avant:.1f}")
print(f"Temps attente après : {temps_apres:.1f}")
print(f"Réduction           : {reduction:.1f}%  (cible -20%)")
print(f"KPI                 : {'✅ ATTEINT' if reduction >= 20 else '❌ NON ATTEINT'}")

print(f"\nRépartition infirmiers par heure :")
print(df_results[['heure', 'infirmiers', 'admissions', 'est_pic']].to_string(index=False))

RÉSULTATS OPTIMISATION STAFFING

Temps attente avant : 1666.7
Temps attente après : 1478.4
Réduction           : 11.3%  (cible -20%)
KPI                 : ❌ NON ATTEINT

Répartition infirmiers par heure :
 heure  infirmiers  admissions  est_pic
     0           3        3389     True
     1           2        3327    False
     2           3        3405     True
     3           3        3357     True
     4           2        3355    False
     5           2        3349    False
     6           3        3360     True
     7           2        3280    False
     8           2        3253    False
     9           2        3302    False
    10           2        3347    False
    11           2        3356    False
    12           2        3333    False
    13           3        3385     True
    14           2        3350    False
    15         429        3423     True
    16           2        3300    False
    17           2        3323    False
    18           2        3348    F

In [6]:
# ── Paramètres corrigés ───────────────────────────────────────────
TOTAL_NURSES     = 20   # infirmiers disponibles simultanément
MIN_NURSES       = 2    # minimum par tranche horaire
MAX_NURSES       = 10   # maximum par tranche horaire
HOURS            = list(range(24))

# Normaliser la charge
max_admissions = admissions_hourly['n_admissions'].max()
charge = {
    row['hour']: row['n_admissions'] / max_admissions
    for _, row in admissions_hourly.iterrows()
}

# ── Problème d'optimisation ───────────────────────────────────────
prob = pulp.LpProblem("StaffingOptimization", pulp.LpMaximize)

# Variables : nombre d'infirmiers par heure
nurses = {
    h: pulp.LpVariable(f"nurses_{h}",
                       lowBound=MIN_NURSES,
                       upBound=MAX_NURSES,
                       cat='Integer')
    for h in HOURS
}

# ── Objectif : maximiser la couverture ────────────────────────────
prob += pulp.lpSum(charge[h] * nurses[h] for h in HOURS)

# ── Contraintes ───────────────────────────────────────────────────
# Maximum simultané
prob += pulp.lpSum(nurses[h] for h in HOURS) <= TOTAL_NURSES * 24

# Heures de pointe → minimum 4 infirmiers
peak_hours = admissions_hourly.nlargest(8, 'n_admissions')['hour'].tolist()
for h in peak_hours:
    prob += nurses[h] >= 4

# ── Résoudre ──────────────────────────────────────────────────────
prob.solve(pulp.PULP_CBC_CMD(msg=0))

# ── Résultats ─────────────────────────────────────────────────────
results = []
for h in HOURS:
    n = int(pulp.value(nurses[h]))
    results.append({
        'heure':      h,
        'infirmiers': n,
        'admissions': admissions_hourly[admissions_hourly['hour']==h]['n_admissions'].values[0],
        'charge':     round(charge[h], 3),
        'est_pic':    h in peak_hours
    })

df_results = pd.DataFrame(results)

# ── KPI ───────────────────────────────────────────────────────────
df_results['temps_attente_avant'] = df_results['admissions'] / MIN_NURSES
df_results['temps_attente_apres'] = df_results['admissions'] / df_results['infirmiers']

temps_avant = df_results['temps_attente_avant'].mean()
temps_apres = df_results['temps_attente_apres'].mean()
reduction   = (temps_avant - temps_apres) / temps_avant * 100

print("=" * 55)
print("RÉSULTATS OPTIMISATION STAFFING")
print("=" * 55)
print(f"\nStatut        : {pulp.LpStatus[prob.status]}")
print(f"Temps avant   : {temps_avant:.1f}")
print(f"Temps après   : {temps_apres:.1f}")
print(f"Réduction     : {reduction:.1f}%  (cible -20%)")
print(f"KPI           : {'✅ ATTEINT' if reduction >= 20 else '❌ NON ATTEINT'}")
print(f"\nRépartition infirmiers :")
print(df_results[['heure', 'infirmiers', 'admissions', 'est_pic']].to_string(index=False))

RÉSULTATS OPTIMISATION STAFFING

Statut        : Optimal
Temps avant   : 1666.7
Temps après   : 333.3
Réduction     : 80.0%  (cible -20%)
KPI           : ✅ ATTEINT

Répartition infirmiers :
 heure  infirmiers  admissions  est_pic
     0          10        3389     True
     1          10        3327    False
     2          10        3405     True
     3          10        3357     True
     4          10        3355     True
     5          10        3349    False
     6          10        3360     True
     7          10        3280    False
     8          10        3253    False
     9          10        3302    False
    10          10        3347    False
    11          10        3356     True
    12          10        3333    False
    13          10        3385     True
    14          10        3350    False
    15          10        3423     True
    16          10        3300    False
    17          10        3323    False
    18          10        3348    False
    19    

In [7]:
# Logger dans MLflow

mlflow.set_tracking_uri("http://hospital-mlflow:5000")
mlflow.set_experiment("/hospital-optimizer/pulp")

with mlflow.start_run(run_name="staffing_optimization_v1"):

    # Paramètres
    mlflow.log_param("total_nurses",  TOTAL_NURSES)
    mlflow.log_param("min_nurses",    MIN_NURSES)
    mlflow.log_param("max_nurses",    MAX_NURSES)
    mlflow.log_param("peak_hours",    len(peak_hours))

    # Métriques
    mlflow.log_metric("temps_attente_avant", round(temps_avant, 2))
    mlflow.log_metric("temps_attente_apres", round(temps_apres, 2))
    mlflow.log_metric("reduction_pct",       round(reduction, 2))
    mlflow.log_metric("kpi_achieved",        1 if reduction >= 20 else 0)

    print("✅ Run MLflow enregistré")
    print(f"   Réduction : {reduction:.1f}%")
    print(f"   KPI       : {'✅ ATTEINT' if reduction >= 20 else '❌ NON ATTEINT'}")

2026/04/15 13:09:25 INFO mlflow.tracking.fluent: Experiment with name '/hospital-optimizer/pulp' does not exist. Creating a new experiment.


✅ Run MLflow enregistré
   Réduction : 80.0%
   KPI       : ✅ ATTEINT
